In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [9]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [ ]:
# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})

Users data

In [ ]:
# users_data = {}
# users_scores = {}

# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})
# mal_client = MALClient(client_id)
# for user in users:
#     user_data = mal_client.get_user_data(user)
#     users_data[user] = user_data
#     scores = mal_client.get_scores(user_data)
#     users_scores[user] = scores

In [10]:
anime_data_client = AnimeDataClient(
    client_id,
    cache_file=PROJECT_ROOT / "anime_cache.json",
)

In [11]:
anime_data = anime_data_client.get_cache()

Build features

In [ ]:
# from anime_features import AnimeFeatureBuilder

# builder = AnimeFeatureBuilder(
#     anime_data,
#     max_tfidf_features=3000,
#     n_svd_components=300
# )

# anime_df = builder.build_features()

# builder.svd_explained_variance

Convert each anime in df to vectors

In [ ]:
# recommender = SimilarityRecommender()
# anime_vectors = recommender.create_anime_vectors(anime_df)
# anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [12]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Tune SVD components

In [13]:
from anime_evaluation import HitRateEvaluator
from anime_features import AnimeFeatureBuilder

svd_component_results = []
n_runs = 100
max_features = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500]
components = [50, 100, 200, 300]
weights_uncertainty = [7.5]
tuning_top_ks = [5, 10]
for n_feature in max_features:
    for component in components:
        builder = AnimeFeatureBuilder(
            anime_data,
            max_tfidf_features=n_feature,
            n_svd_components=component,
        )

        component_anime_df = builder.build_features()

        recommender = SimilarityRecommender()
        recommender.create_anime_vectors(component_anime_df)
        component_anime_df_scaled = recommender.anime_df_scaled

        hitman = HitRateEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )

        (
            bayesian_results,
            bayesian_summary,
            best_bayesian_weights,
            baseline_results,
            baseline_summary,
        ) = hitman.tune_bayesian_uncertainty(
            weights=weights_uncertainty,
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
        )


        average_metrics = bayesian_summary.merge(
            baseline_summary,
            on="k",
            how="left",
        ).rename(columns={"uncertainty_weight": "bayesian_uncertainty_weight"})
        average_metrics["component"] = component
        average_metrics["n_feature"] = n_feature
        average_metrics["svd_explained_variance"] = builder.svd_explained_variance

        svd_component_results.append(average_metrics)

svd_component_summary = (
    pd.concat(svd_component_results, ignore_index=True)
    .sort_values(["k", "avg_precision_at_k"], ascending=[True, False])
)

svd_component_summary

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,component,n_feature,svd_explained_variance
46,7.5,5,0.522,0.190470,0.084194,0.030721,2.61,0.098,0.128692,0.015806,0.020757,0.49,300,3000,0.410064
62,7.5,5,0.522,0.186179,0.084194,0.030029,2.61,0.098,0.128692,0.015806,0.020757,0.49,300,4000,0.373075
52,7.5,5,0.520,0.186407,0.083871,0.030066,2.60,0.098,0.128692,0.015806,0.020757,0.49,200,3500,0.298001
54,7.5,5,0.520,0.186407,0.083871,0.030066,2.60,0.098,0.128692,0.015806,0.020757,0.49,300,3500,0.388608
44,7.5,5,0.518,0.188765,0.083548,0.030446,2.59,0.098,0.128692,0.015806,0.020757,0.49,200,3000,0.315293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,7.5,10,0.269,0.109816,0.086774,0.035425,2.69,0.066,0.062312,0.021290,0.020101,0.66,50,1500,0.170254
3,7.5,10,0.268,0.106249,0.086452,0.034274,2.68,0.066,0.062312,0.021290,0.020101,0.66,100,500,0.435331
9,7.5,10,0.266,0.107516,0.085806,0.034682,2.66,0.066,0.062312,0.021290,0.020101,0.66,50,1000,0.203520
5,7.5,10,0.262,0.102277,0.084516,0.032993,2.62,0.066,0.062312,0.021290,0.020101,0.66,200,500,0.659237


In [16]:
svd_component_summary.head(10)

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,component,n_feature,svd_explained_variance
46,7.5,5,0.522,0.190470,0.084194,0.030721,2.61,0.098,0.128692,0.015806,0.020757,0.49,300,3000,0.410064
62,7.5,5,0.522,0.186179,0.084194,0.030029,2.61,0.098,0.128692,0.015806,0.020757,0.49,300,4000,0.373075
52,7.5,5,0.520,0.186407,0.083871,0.030066,2.60,0.098,0.128692,0.015806,0.020757,0.49,200,3500,0.298001
54,7.5,5,0.520,0.186407,0.083871,0.030066,2.60,0.098,0.128692,0.015806,0.020757,0.49,300,3500,0.388608
44,7.5,5,0.518,0.188765,0.083548,0.030446,2.59,0.098,0.128692,0.015806,0.020757,0.49,200,3000,0.315293
60,7.5,5,0.518,0.186613,0.083548,0.030099,2.59,0.098,0.128692,0.015806,0.020757,0.49,200,4000,0.285609
38,7.5,5,0.516,0.188947,0.083226,0.030475,2.58,0.098,0.128692,0.015806,0.020757,0.49,300,2500,0.437740
30,7.5,5,0.510,0.197714,0.082258,0.031889,2.55,0.098,0.128692,0.015806,0.020757,0.49,300,2000,0.474341
22,7.5,5,0.508,0.203842,0.081935,0.032878,2.54,0.098,0.128692,0.015806,0.020757,0.49,300,1500,0.528033
36,7.5,5,0.508,0.191580,0.081935,0.030900,2.54,0.098,0.128692,0.015806,0.020757,0.49,200,2500,0.338016


In [14]:
best_svd_components = (
    svd_component_summary
    .sort_values(["k", "avg_precision_at_k", "avg_hit_rate"], ascending=[True, False, False])
    .groupby("k")
    .head(1)
)

best_svd_components

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,component,n_feature,svd_explained_variance
46,7.5,5,0.522,0.19047,0.084194,0.030721,2.61,0.098,0.128692,0.015806,0.020757,0.49,300,3000,0.410064
55,7.5,10,0.319,0.12366,0.102903,0.039890,3.19,0.066,0.062312,0.021290,0.020101,0.66,300,3500,0.388608


## Results

This tuning run evaluated `max_tfidf_features` from **500** to **4500** and SVD components from **50** to **300**, using **100 runs**, uncertainty weight **7.5**, and `random_state=42` for reproducible holdout splits.

| k | Best max TF-IDF features | Best SVD components | Avg precision@k | Std precision@k | Avg hit rate | Avg hits | Baseline precision@k | SVD explained variance |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 5 | 3000 | 300 | 0.522 | 0.190 | 0.0842 | 2.61 | 0.098 | 0.4101 |
| 10 | 3500 | 300 | 0.319 | 0.124 | 0.1029 | 3.19 | 0.066 | 0.3886 |

For `k=5`, the best setting was **3000 TF-IDF features** with **300 SVD components**. The older `3000/200` setting was still very close, but `3000/300` gave the strongest top-5 precision in this run.

For `k=10`, the best setting was **3500 TF-IDF features** with **300 SVD components**. Since top-5 quality is the more important recommendation cutoff here, prefer the `3000/300` setting as the default unless optimizing specifically for longer top-10 lists.

Conclusion: use **`max_tfidf_features=3000`** and **`n_svd_components=300`** as the default feature configuration.
